In [0]:
%run ./transform_data

# Paramètres
date_debut = "2025-01-01"
aujourdhui = datetime.now()
today_date = aujourdhui.date()
current_hour = aujourdhui.hour
current_minute = aujourdhui.minute

# Générer la liste des dates entre date_debut et aujourd'hui
df_dates = (
    spark.createDataFrame([(1,)], ["dummy"])
    .select(F.explode(F.sequence(F.to_date(F.lit(date_debut)), F.current_date(), F.expr("interval 1 day"))).alias("date"))
)

# Générer toutes les minutes dans une journée
df_minutes = (
    spark.range(0, 24*60)
    .withColumn("heure", F.floor(F.col("id") / 60))
    .withColumn("minute", F.col("id") % 60)
    .drop("id")
)

# Produit cartésien
df = df_dates.crossJoin(df_minutes)

# Filtrer selon la logique (tous les jours complets sauf aujourd'hui)
df_filtered = df.filter(
    (F.col("date") < F.lit(str(today_date))) |
    (
        (F.col("date") == F.lit(str(today_date))) &
        (
            (F.col("heure") < current_hour) |
            ((F.col("heure") == current_hour) & (F.col("minute") <= current_minute))
        )
    )
)

# Construire datetime (timestamp) + colonne datetime_hour (troncature à l'heure)
dim_calendar = df_filtered.withColumn(
    "datetime",
    F.to_timestamp(
        F.concat_ws(
            " ",
            F.date_format("date", "yyyy-MM-dd"),
            F.format_string("%02d:%02d:00", F.col("heure"), F.col("minute"))
        ),
        "yyyy-MM-dd HH:mm:ss"
    )
).withColumn(
    "datetime_hour",
    F.date_trunc("hour", F.col("datetime"))   # <-- ici on garde l'heure seulement (minutes=00)
).drop("heure", "minute").orderBy("datetime")



In [0]:

# Paramètre
date_debut = "2025-01-01"

dim_calendar = (
    spark.createDataFrame([(1,)], ["dummy"])
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit(date_debut)),
                F.current_date(),
                F.expr("interval 1 day")
            )
        ).alias("date_simple")
    )
    .withColumn("month_name", F.date_format("date_simple", "MMMM"))
    .withColumn("year_name", F.year("date_simple"))
    .orderBy("date_simple")
)


Ingestion des données dans la table cible

In [0]:
current_process= "dim_calendar"

In [0]:
target_dim_calendar = current_catalog +"."+current_schema+"."+current_process
print(target_dim_calendar)

In [0]:
all_columns =  dim_calendar.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'date_simple']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    dim_calendar, 
    target_dim_calendar, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )